<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week3_unsupervised/Week3_Lesson6_Workshop.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛒 IB CS — Неделя 3 · Урок 6 (Семинар)
## Workshop "Unsupervised in Action"
### A4.3.4 + A4.3.5 + A4.3.10 на практике

> ⏱ Длительность: 2 академических часа.
> 💻 Платформа: Google Colab
> 🎯 Цель: пройти **3 unsupervised задачи** на реальных датасетах + сравнить модели через cross-validation.

---

### 📋 План семинара

| Часть | Тема | Время |
|---|---|---|
| 1 | **K-means** customer segmentation + Elbow method | 25 мин |
| 2 | **DBSCAN** на non-spherical данных | 20 мин |
| 3 | **Apriori** market basket analysis | 30 мин |
| 4 | **Model comparison** через cross-validation | 25 мин |
| 5 | IB-style вывод | 10 мин |

---

### ⚙️ Установка библиотек

> Если в Colab `mlxtend` не установлен, выполните в первой ячейке: `!pip install mlxtend`


In [ ]:
# Если нужно — установите mlxtend (для Apriori)
# !pip install mlxtend --quiet

# Импорты
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

np.random.seed(2027)
sns.set_style('whitegrid')
print("✅ Все библиотеки готовы")

## Часть 1 · Customer Segmentation с K-means

### Сценарий
> Магазин одежды хочет сегментировать клиентов по 2 признакам:
> - **Annual income** (тыс. $)
> - **Spending score** (1–100; "склонность к покупкам")

Чтобы потом строить **разные маркетинговые стратегии** для каждой группы.


In [ ]:
# Синтетический датасет: 5 типов клиентов
n_per_group = 50
customers = []
labels_true = []

# Группа 1: высокий доход + высокие траты ("Champions")
customers.append(np.random.normal([85, 80], [10, 8], (n_per_group, 2)))
# Группа 2: низкий доход + низкие траты ("Budget")
customers.append(np.random.normal([25, 25], [8, 8], (n_per_group, 2)))
# Группа 3: высокий доход + низкие траты ("Misers")
customers.append(np.random.normal([85, 25], [10, 8], (n_per_group, 2)))
# Группа 4: низкий доход + высокие траты ("Risky")
customers.append(np.random.normal([25, 80], [8, 8], (n_per_group, 2)))
# Группа 5: средние ("Average")
customers.append(np.random.normal([55, 55], [10, 10], (n_per_group, 2)))

X_cust = np.vstack(customers)
df_cust = pd.DataFrame(X_cust, columns=['Annual Income (k$)', 'Spending Score'])

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df_cust['Annual Income (k$)'], df_cust['Spending Score'],
           c='gray', s=50, alpha=0.6, edgecolor='black')
ax.set_xlabel('Annual Income (k$)', fontsize=11)
ax.set_ylabel('Spending Score (1-100)', fontsize=11)
ax.set_title('Клиенты ДО кластеризации — мы НЕ знаем меток!',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"📊 Всего клиентов: {len(df_cust)}")

### 🎯 TRY IT #1 — Сколько кластеров вы видите?

Прежде чем смотреть на алгоритм, **угадайте**:
- Сколько естественных групп клиентов?
- Где центроиды (примерно)?

> 💡 Это важно — в реальном IA вы должны **обосновать** выбор K не только Elbow, но и доменной экспертизой.


### Шаг 1 · Elbow Method — выбор K

In [ ]:
# Перебираем K от 1 до 10
ks = range(1, 11)
inertias = []
silhouettes = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cust)
    inertias.append(km.inertia_)
    if k > 1:
        silhouettes.append(silhouette_score(X_cust, km.labels_))
    else:
        silhouettes.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ks, inertias, 'o-', linewidth=2, markersize=10, color='steelblue')
axes[0].axvline(5, color='red', linestyle='--', alpha=0.7, label='Elbow at K=5')
axes[0].set_xlabel('K (число кластеров)', fontsize=11)
axes[0].set_ylabel('Inertia (sum of squared distances)', fontsize=11)
axes[0].set_title('Elbow Method', fontsize=12, fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ks, silhouettes, 'o-', linewidth=2, markersize=10, color='orange')
axes[1].axvline(5, color='red', linestyle='--', alpha=0.7, label='Best at K=5')
axes[1].set_xlabel('K', fontsize=11)
axes[1].set_ylabel('Silhouette Score (higher = better)', fontsize=11)
axes[1].set_title('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print(f"💎 Best K по Elbow: ~5 (где замедляется падение inertia)")
print(f"💎 Best K по Silhouette: {np.nanargmax(silhouettes) + 1}")

> 💎 **СЕКРЕТ для IA:** показывайте **ОБА графика** — Elbow + Silhouette. Это даёт **2 независимых** обоснования выбора K. Экзаменатор это ценит.

### Шаг 2 · Финальная сегментация с K=5


In [ ]:
# Финальная модель
best_k = 5
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_cust['cluster'] = km_final.fit_predict(X_cust)

# Имена сегментам (бизнес-интерпретация!)
segment_names = {}
for c in range(best_k):
    centroid = km_final.cluster_centers_[c]
    income, spending = centroid
    if income > 60 and spending > 60:
        segment_names[c] = 'Champions (high income, high spending)'
    elif income < 40 and spending < 40:
        segment_names[c] = 'Budget (low income, low spending)'
    elif income > 60 and spending < 40:
        segment_names[c] = 'Misers (high income, low spending)'
    elif income < 40 and spending > 60:
        segment_names[c] = 'Risky (low income, high spending)'
    else:
        segment_names[c] = 'Average (middle)'

# Визуализация
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#FF6B6B', '#4ECDC4', '#FFD93D', '#6BCB77', '#A78BFA']
for c in range(best_k):
    mask = df_cust['cluster'] == c
    ax.scatter(df_cust.loc[mask, 'Annual Income (k$)'],
               df_cust.loc[mask, 'Spending Score'],
               c=colors[c], s=60, alpha=0.7, edgecolor='black',
               label=segment_names[c])
ax.scatter(km_final.cluster_centers_[:,0], km_final.cluster_centers_[:,1],
           marker='X', s=300, c='black', edgecolor='white', linewidth=2,
           label='Centroids')
ax.set_xlabel('Annual Income (k$)', fontsize=11)
ax.set_ylabel('Spending Score', fontsize=11)
ax.set_title(f'Customer Segmentation (K-means, K={best_k})',
             fontsize=12, fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
plt.tight_layout(); plt.show()

# Таблица: какая стратегия для каждого сегмента?
print("\n=== Бизнес-стратегии для каждого сегмента ===")
strategies = pd.DataFrame({
    'Сегмент': [segment_names[c] for c in range(best_k)],
    'Размер': [(df_cust['cluster']==c).sum() for c in range(best_k)],
    'Стратегия': [
        'VIP-программа, эксклюзивные продукты',
        'Скидки, бюджетные коллекции',
        'Премиум-маркетинг, имидж',
        'Кредитные программы, осторожный апсейл',
        'Стандартный маркетинг'
    ]
})
print(strategies.to_string(index=False))

> 💎 **СЕКРЕТ для IA:** после кластеризации **обязательно** дайте **бизнес-интерпретацию**. Просто "5 кластеров" = слабо. "Champions, Budget, Misers, Risky, Average + стратегия для каждого" = сильный IA.


## Часть 2 · DBSCAN — когда K-means не справляется

### Сценарий
> Анализ GPS-треков туристов в Барселоне. Скопления формируются вдоль улиц (вытянутые формы) + есть **outliers** (одинокие туристы).


In [ ]:
# Генерируем данные: 2 "полумесяца" + шум
X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)
# Добавим outliers
outliers = np.random.uniform(-1.5, 2.5, (15, 2))
X_with_noise = np.vstack([X_moons, outliers])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Исходные данные
axes[0].scatter(X_with_noise[:,0], X_with_noise[:,1], c='gray', s=30, alpha=0.7)
axes[0].set_title('Исходные данные с outliers', fontsize=11)

# K-means — ПРОВАЛ
km = KMeans(n_clusters=2, random_state=42, n_init=10)
km_labels = km.fit_predict(X_with_noise)
axes[1].scatter(X_with_noise[:,0], X_with_noise[:,1], c=km_labels, cmap='viridis',
                s=30, alpha=0.7)
axes[1].scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1],
                marker='X', s=200, c='red', edgecolor='black')
axes[1].set_title('K-means: разрезает полумесяцы\n(outliers попадают в кластер)',
                  fontsize=11, fontweight='bold', color='darkred')

# DBSCAN — успех
db = DBSCAN(eps=0.2, min_samples=5)
db_labels = db.fit_predict(X_with_noise)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = list(db_labels).count(-1)

# Чёрные точки = noise
mask_noise = db_labels == -1
mask_clusters = ~mask_noise
axes[2].scatter(X_with_noise[mask_clusters,0], X_with_noise[mask_clusters,1],
                c=db_labels[mask_clusters], cmap='viridis', s=30, alpha=0.7)
axes[2].scatter(X_with_noise[mask_noise,0], X_with_noise[mask_noise,1],
                c='black', s=80, marker='x', linewidth=2, label=f'Noise ({n_noise})')
axes[2].set_title(f'DBSCAN: {n_clusters} кластеров + {n_noise} noise',
                  fontsize=11, fontweight='bold', color='darkgreen')
axes[2].legend()

plt.tight_layout(); plt.show()

### 🎯 TRY IT #2 — Поэкспериментируйте с параметрами DBSCAN

В коде ниже измените `eps` и `min_samples`. Что происходит?
- `eps=0.1` → слишком мало кластеров
- `eps=0.5` → всё в одном кластере
- `min_samples=20` → больше точек становятся noise


In [ ]:
# Эксперимент с параметрами DBSCAN
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

configs = [(0.1, 5), (0.2, 5), (0.4, 5)]
for ax, (eps, min_s) in zip(axes, configs):
    db = DBSCAN(eps=eps, min_samples=min_s)
    labels = db.fit_predict(X_with_noise)
    n_clust = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)

    mask_noise = labels == -1
    ax.scatter(X_with_noise[~mask_noise,0], X_with_noise[~mask_noise,1],
               c=labels[~mask_noise], cmap='viridis', s=30, alpha=0.7)
    ax.scatter(X_with_noise[mask_noise,0], X_with_noise[mask_noise,1],
               c='black', s=80, marker='x', linewidth=2)
    ax.set_title(f'eps={eps}, min_samples={min_s}\n→ {n_clust} clusters, {n_noise} noise',
                 fontsize=11)

plt.suptitle('Влияние eps на DBSCAN', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## Часть 3 · Apriori — Market Basket Analysis

### Сценарий
> Продуктовый магазин хочет понять, **какие товары покупают вместе**, чтобы:
> - Размещать их рядом на полках
> - Делать bundle-скидки
> - Рекомендовать в онлайн-корзине


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Синтетические транзакции (50 покупок)
transactions = [
    ['milk', 'bread', 'butter'],
    ['milk', 'bread', 'eggs'],
    ['milk', 'bread'],
    ['bread', 'butter'],
    ['milk', 'butter', 'eggs'],
    ['bread', 'butter', 'jam'],
    ['milk', 'bread', 'butter'],
    ['eggs', 'cheese'],
    ['milk', 'bread', 'butter', 'eggs'],
    ['bread', 'jam'],
    ['milk', 'eggs'],
    ['bread', 'butter', 'cheese'],
    ['milk', 'bread', 'jam'],
    ['cheese', 'wine'],
    ['milk', 'bread', 'butter'],
    ['bread', 'butter', 'eggs'],
    ['milk', 'cheese'],
    ['bread', 'jam', 'butter'],
    ['wine', 'cheese', 'bread'],
    ['milk', 'bread', 'eggs', 'cheese'],
    ['bread', 'butter'],
    ['milk', 'bread'],
    ['milk', 'butter'],
    ['eggs', 'milk'],
    ['bread', 'butter', 'cheese'],
    ['wine', 'cheese'],
    ['milk', 'bread', 'butter'],
    ['bread', 'butter', 'eggs'],
    ['milk', 'bread', 'jam'],
    ['cheese', 'wine', 'bread'],
    ['milk', 'eggs', 'butter'],
    ['bread', 'butter'],
    ['milk', 'bread', 'butter', 'eggs'],
    ['jam', 'bread', 'butter'],
    ['milk', 'cheese', 'wine'],
    ['bread', 'butter', 'jam'],
    ['eggs', 'milk', 'bread'],
    ['cheese', 'wine', 'bread', 'butter'],
    ['milk', 'bread'],
    ['butter', 'jam', 'bread'],
    ['milk', 'bread', 'butter'],
    ['eggs', 'butter'],
    ['cheese', 'milk'],
    ['wine', 'cheese', 'bread', 'butter'],
    ['milk', 'bread', 'jam', 'butter'],
    ['bread', 'butter', 'eggs', 'cheese'],
    ['milk', 'butter', 'jam'],
    ['cheese', 'bread', 'butter'],
    ['milk', 'eggs', 'cheese'],
    ['bread', 'butter', 'milk', 'eggs'],
]

print(f"📊 Всего транзакций: {len(transactions)}")

# One-hot encoding
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_trans = pd.DataFrame(te_ary, columns=te.columns_)
print(f"📊 Уникальных товаров: {len(te.columns_)}")
print(f"\nПервые 5 транзакций (one-hot):")
print(df_trans.head().astype(int))

In [ ]:
# Step 1: Найти frequent itemsets с min_support = 0.2 (20%)
frequent_itemsets = apriori(df_trans, min_support=0.2, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print("=== Frequent Itemsets (support >= 0.20) ===")
print(frequent_itemsets.sort_values(['length', 'support'], ascending=[True, False]).to_string(index=False))

In [ ]:
# Step 2: Создать association rules
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.5)

# Отображаем самые интересные правила (по lift)
print("=== Top 10 Association Rules (sorted by Lift) ===")
display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
print(rules[display_cols].sort_values('lift', ascending=False).head(10).round(3).to_string(index=False))

In [ ]:
# Визуализация: scatter plot Support vs Confidence, размер = Lift
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(rules['support'], rules['confidence'],
                     s=rules['lift']*300, alpha=0.5,
                     c=rules['lift'], cmap='RdYlGn',
                     edgecolor='black')
ax.set_xlabel('Support', fontsize=11)
ax.set_ylabel('Confidence', fontsize=11)
ax.set_title('Association Rules: Support × Confidence (size/color = Lift)',
             fontsize=12, fontweight='bold')
plt.colorbar(scatter, label='Lift')
ax.grid(alpha=0.3)

# Подпишем топ-3 правила
top3 = rules.nlargest(3, 'lift')
for _, row in top3.iterrows():
    label = f"{list(row['antecedents'])} → {list(row['consequents'])}"
    ax.annotate(label, (row['support'], row['confidence']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=9, fontweight='bold')

plt.tight_layout(); plt.show()

### 🎯 TRY IT #3 — Бизнес-интерпретация правил

Глядя на топ-правила, ответьте:

1. Какие 2 товара имеют **самый высокий lift**? Что это значит для магазина?
2. Какое правило самое **надёжное** (наивысший confidence)?
3. Какое правило имеет **самый высокий support** (часто встречается)?

> 💎 На реальном экзамене (Section B) могут показать таблицу правил и спросить: *"Which rule should the marketing team prioritize? Justify."* Ответ: смотреть на **lift** (сила связи) + **support** (достаточно ли частое явление).


## Часть 4 · Model Selection через Cross-Validation

> Возвращаемся к **supervised learning** с уроков прошлой недели — но теперь делаем сравнение **правильно** (через CV).

### Сценарий
> Сравним **3 модели** для классификации с помощью **5-fold cross-validation** + расширенными метриками.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

# Бинарная классификация: 500 объектов, 8 фич
X_clf, y_clf = make_classification(n_samples=500, n_features=8, n_informative=5,
                                    n_redundant=2, random_state=42)

# 3 модели для сравнения (KNN с pipeline для нормализации)
models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()),
                                      ('lr', LogisticRegression(max_iter=1000))]),
    'Decision Tree (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'KNN (k=7)': Pipeline([('scaler', StandardScaler()),
                            ('knn', KNeighborsClassifier(n_neighbors=7))]),
}

# Cross-validate по всем 4 метрикам сразу
metrics = ['accuracy', 'precision', 'recall', 'f1']
results = []

for name, model in models.items():
    cv = cross_validate(model, X_clf, y_clf, cv=5, scoring=metrics)
    row = {'Model': name}
    for m in metrics:
        scores = cv[f'test_{m}']
        row[f'{m}_mean'] = scores.mean()
        row[f'{m}_std']  = scores.std()
    results.append(row)

results_df = pd.DataFrame(results)
print("=== Cross-Validation Results (5-fold) ===\n")
for _, row in results_df.iterrows():
    print(f"{row['Model']}:")
    for m in metrics:
        print(f"  {m:10s} = {row[f'{m}_mean']:.4f} ± {row[f'{m}_std']:.4f}")
    print()

In [ ]:
# Визуализация: bar chart с error bars
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(results_df))
width = 0.2
colors = ['#FF6B6B', '#4ECDC4', '#FFD93D', '#6BCB77']

for i, m in enumerate(metrics):
    means = results_df[f'{m}_mean'].values
    stds  = results_df[f'{m}_std'].values
    ax.bar(x + i*width, means, width, yerr=stds, capsize=4,
           label=m, color=colors[i], edgecolor='black')

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
ax.set_ylabel('Score (5-fold CV)', fontsize=11)
ax.set_title('Model Comparison — Cross-Validation Results\n(error bars = std across folds)',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

### 🎯 TRY IT #4 — Выбор лучшей модели

По графику:
1. Какая модель **лучшая по F1**?
2. У какой модели **наименьший разброс** (std)? → более надёжная.
3. Если бизнес-приоритет = **высокий recall** (например, медицина) — какая модель?
4. Если бизнес-приоритет = **интерпретируемость** — какая модель?

> 💎 **СЕКРЕТ:** на IA / Section B **никогда** не выбирайте модель только по одной метрике. Обоснование должно учитывать **минимум 3 критерия** (F1 + std + интерпретируемость + контекст).


## Часть 5 · IB-style итог

### 📝 Шаблон для отчёта IA по unsupervised learning

```
1. PROBLEM TYPE: Unsupervised learning (нет меток в данных)
   - Цель: customer segmentation / pattern discovery

2. ALGORITHM CHOICE:
   - K-means для известного K + сферические кластеры
   - DBSCAN для произвольной формы + handling outliers
   - Hierarchical для иерархических данных без K
   - Apriori для co-occurrence patterns

3. HYPERPARAMETER SELECTION:
   - K via Elbow method + Silhouette score
   - ε и minPts для DBSCAN — через эксперименты
   - Min support и confidence для Apriori — через бизнес-контекст

4. EVALUATION:
   - Visualization (scatter plots с цветом по кластеру)
   - Silhouette score
   - Бизнес-интерпретация кластеров (даём имена!)

5. RECOMMENDATIONS:
   - Конкретные действия для каждого сегмента
   - Trade-offs между алгоритмами
```

---

### 💎 Финальные секреты семинара

1. **Elbow + Silhouette вместе** → надёжный выбор K.
2. **DBSCAN видит то, что K-means не может** — non-spherical, outliers.
3. **Lift > 1 — самая важная** метрика в association rules (показывает силу связи).
4. **Бизнес-имена сегментам** = +1 балл на IA.
5. **Cross-validation с std** (error bars) — golden standard.
6. **mlxtend** — стандартная библиотека для Apriori в Python.

---

### 🏠 ДЗ (см. `Week3_HW2_Practice.ipynb`)

> Полный mini-IA: customer segmentation + market basket + model selection report.
